# Expected Returns Summary Analytics
#
# Comprehensive analysis of the unified `expected_returns_summary` dataset covering
# 7 probabilistic models across ~5,500 stocks:
# - **Monte Carlo Simulation** — Upside/downside distributions, VaR, risk-reward
# - **Kalman Filtered Targets** — Noise-reduced signals, gain & variance diagnostics
# - **Price Target Achievement** — Probability-weighted returns, analyst conviction
# - **Earnings Beat Analysis** — Bayesian posterior beat probabilities, momentum signals
# - **Credit Risk** — Distress probability, Altman Z-score, survival analysis
# - **Dividend Safety** — Cut probability, FCF coverage, payout sustainability
# - **Accounting Anomaly Detection** — Anomaly scores, flag counts, sector-relative risk
# - **Cross-Model Agreement** — Composite scores, signal alignment, quality tiers


## 1. Setup & Data Loading


In [2]:
import warnings
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

warnings.filterwarnings("ignore")

PLOTLY_TEMPLATE = "plotly_dark"
COLORS = px.colors.qualitative.Dark24


,ticker,name,country,exchange,sector,industry,achievement_probability,upside_potential,price_target_spread_pct,analyst_conviction,eps_revision_momentum,analyst_rating_normalized,expected_return_prob_weighted,confidence_level
0,AER,AerCap Holdings N.V.,IE,NYSE,Industrials,Trading Companies and Distributors,0.83,12.086814,13.375796,88.888889,0.038825,89.00,10.032055,High
1,PM,Philip Morris International Inc.,US,NYSE,Consumer Staples,Tobacco,0.90,-1.537115,23.333333,63.157895,0.006085,77.75,-1.383404,Medium
2,UAA,Under Armour Inc.,US,NYSE,Consumer Discretionary,Textiles Apparel and Luxury Goods,0.85,-27.248677,174.545455,12.000000,0.128545,57.00,-23.161376,Low
3,GGD,GoGold Resources Inc.,CA,TSX,Materials,Metals and Mining,0.54,59.235669,0.000000,100.000000,0.604565,100.00,31.987261,Low
4,NVT,nVent Electric plc,GB,NYSE,Industrials,Electrical Equipment,0.62,15.411932,59.200000,92.857143,0.039150,92.75,9.555398,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6414,INRN,Interroll Holding AG,CH,SWX,Industrials,Machinery,0.39,33.333333,26.576923,57.142857,0.006545,75.00,13.000000,Medium
6415,LOTB,Lotus Bakeries NV,BE,ENXTBR,Consumer Staples,Food Products,0.90,-6.250000,35.353535,60.000000,0.015580,82.50,-5.625000,Low
6416,YSN,secunet Security Networks Aktiengesellschaft,DE,XTRA,Information Technology,IT Services,0.65,25.183374,21.484375,75.000000,0.091300,87.50,16.369193,Medium
6417,KOMN,Komax Holding AG,CH,SWX,Industrials,Machinery,0.84,-2.139800,32.069971,0.000000,0.000000,55.00,-1.797432,Medium


### Load Data


In [4]:
returns_df = pd.read_csv("expected_returns_summary.csv", low_memory=False, usecols=range(203))

# Clean categorical columns (drop rare misaligned rows)
cat_sectors = [
    "Industrials", "Information Technology", "Consumer Discretionary",
    "Health Care", "Materials", "Consumer Staples", "Energy",
    "Communication Services", "Utilities", "Financials", "Real Estate",
]
returns_df = returns_df[returns_df["sector"].isin(cat_sectors)].copy()

# Coerce numeric columns
numeric_cols = [
    "last_price", "expected_upside_mc", "implied_return_mc", "price_target_mc",
    "var_5_pct", "prob_positive_upside", "risk_reward_ratio", "upside_std", "pt_spread",
    "implied_return_kalman", "expected_upside_kalman", "price_target_kalman",
    "kalman_variance", "kalman_gain", "signal_strength",
    "expected_upside_pt", "implied_return_pt", "price_target_prob_weighted",
    "achievement_probability", "analyst_conviction", "bullish_pct",
    "posterior_beat_prob", "confidence_score", "model_confidence", "map_estimate",
    "prob_beat_given_momentum", "continuation_probability", "mean_reversion_probability",
    "prediction_confidence", "momentum_signal", "volatility_regime_score",
    "technical_adjustment", "base_posterior_mean", "resampled_posterior_mean",
    "distress_probability", "altman_z_score", "altman_z_trend",
    "liquidity_stress_score", "cash_runway_months", "survival_probability",
    "ruin_probability", "wealth_buffer", "beta_stability_score",
    "balance_sheet_strength", "wc_efficiency_score", "distress_risk_score",
    "dividend_cut_probability", "fcf_dividend_coverage", "payout_ratio",
    "dividend_consistency", "safety_score", "yield_vs_5y_avg",
    "accounting_anomaly_score", "sector_relative_anomaly", "anomaly_severity_score",
    "anomaly_risk_rank", "sector_anomaly_percentile", "anomaly_conditional_probability",
    "composite_score", "weighted_agreement", "agreement_score",
    "market_cap", "enterprise_value", "forward_revenue_growth",
    "gross_margin_pct", "buyback_yield", "pe_forward_discount",
    "implied_return_mc_zscore", "implied_return_kalman_zscore", "implied_return_pt_zscore",
    "implied_return_mc_pctile", "implied_return_kalman_pctile", "implied_return_pt_pctile",
    "price_target_mc_pctile", "price_target_kalman_pctile",
    "analyst_rating", "analyst_rating_normalized", "price_target_count",
    "eps_revision_momentum_y", "forward_pe_vs_sector_proxy",
    "price_momentum_1m", "price_momentum_3m",
    "debt_maturity_risk_x", "debt_3y_cagr",
    "impairment_risk_score", "quality_frequency_score",
]
for col in numeric_cols:
    if col in returns_df.columns:
        returns_df[col] = pd.to_numeric(returns_df[col], errors="coerce")

print(f"✅ Loaded {returns_df.shape[0]:,} stocks × {returns_df.shape[1]} columns")
print(f"   Sectors: {returns_df['sector'].nunique()}")
print(f"   Size classes: {returns_df['size_class'].value_counts().to_dict()}")


,isin,ticker,name,industry,sector,trading_country,region,country,exchange,analyst_bullish_pct,...,pt_median_momentum_1m,pt_median_momentum_3m,pt_acceleration_short,pt_acceleration_long,pt_consensus_convergence,analyst_coverage_change_1m,analyst_coverage_change_3m,analyst_coverage_change_1y,pt_vs_price_momentum,analyst_coverage_trend
0,US67066G1040,NVDA,NVIDIA Corporation,Semiconductors and Semiconductor Equipment,Information Technology,US,United States and Canada,US,NasdaqGS,93.650794,...,0.000000,0.111111,-0.100153,-0.356934,0.263111,1,2,4,0.116303,0.031897
1,US74006W2070,PRAX,Praxis Precision Medicines Inc.,Biotechnology,Health Care,US,United States and Canada,US,NasdaqGS,87.500000,...,0.145125,0.819820,-0.704158,-1.472271,-0.951102,1,2,6,0.064671,0.131250
2,US0378331005,AAPL,Apple Inc.,Technology Hardware Storage and Peripherals,Information Technology,US,United States and Canada,US,NasdaqGS,61.702128,...,0.000000,0.083032,-0.026305,-0.116889,-0.014019,0,0,1,0.012339,0.018293
3,US2681582019,DVAX,Dynavax Technologies Corporation,Biotechnology,Health Care,US,United States and Canada,US,NasdaqGS,33.333333,...,0.000000,0.000000,0.035088,0.084912,0.480000,0,-2,-2,-0.315846,-0.316667
4,US02079K3059,GOOGL,Alphabet Inc.,Interactive Media and Services,Communication Services,US,United States and Canada,US,NasdaqGS,89.062500,...,0.151515,0.155015,-0.056221,-0.532126,-0.118085,2,2,8,0.025202,0.040179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6415,TN0007140015,ASSAD,L'Accumulateur Tunisien Assad SA,Electrical Equipment,Industrials,TN,Africa / Middle East,TN,BVMT,0.000000,...,0.000000,0.018779,-0.018779,-2.597887,0.000000,0,0,0,-0.239099,0.000000
6416,BRTOKYACNOR5,TOKY3,Grupo Toky S.A.,Specialty Retail,Consumer Discretionary,BR,Latin America and Caribbean,BR,BOVESPA,0.000000,...,0.000000,0.000000,0.000000,0.361702,0.000000,0,0,-1,0.639344,0.000000
6417,BRTCSAACNOR3,TCSA3,Tecnisa S.A.,Household Durables,Consumer Discretionary,BR,Latin America and Caribbean,BR,BOVESPA,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0,-0.219512,0.000000
6418,NGCILEASING2,CILEASING,C & I Leasing Plc,Trading Companies and Distributors,Industrials,NG,Africa / Middle East,NG,NGSE,0.000000,...,0.000000,0.000000,0.000000,NaN,0.000000,0,0,1,-0.260927,0.000000


## 2. Dataset Overview & Summary Statistics


In [ ]:
# High-level summary of key model outputs
summary_cols = {
    "Monte Carlo": ["expected_upside_mc", "implied_return_mc", "prob_positive_upside", "var_5_pct",
                    "risk_reward_ratio"],
    "Kalman Filter": ["expected_upside_kalman", "implied_return_kalman", "kalman_variance", "kalman_gain",
                      "signal_strength"],
    "Price Target": ["expected_upside_pt", "implied_return_pt", "achievement_probability", "analyst_conviction",
                     "bullish_pct"],
    "Earnings Beat": ["posterior_beat_prob", "confidence_score", "model_confidence", "momentum_signal",
                      "prediction_confidence"],
    "Credit Risk": ["distress_probability", "altman_z_score", "survival_probability", "liquidity_stress_score",
                    "balance_sheet_strength"],
    "Dividend Safety": ["dividend_cut_probability", "fcf_dividend_coverage", "payout_ratio", "dividend_consistency",
                        "safety_score"],
    "Accounting Anomaly": ["accounting_anomaly_score", "sector_relative_anomaly", "anomaly_severity_score",
                           "anomaly_conditional_probability", "impairment_risk_score"],
}

for model_name, cols in summary_cols.items():
    valid_cols = [c for c in cols if c in returns_df.columns and returns_df[c].notna().sum() > 0]
    if valid_cols:
        print(f"\n{'=' * 60}")
        print(f"  {model_name} Model — Key Statistics")
        print(f"{'=' * 60}")
        display(returns_df[valid_cols].describe().round(4))


In [ ]:
# Data completeness heatmap
all_model_cols = [c for cols in summary_cols.values() for c in cols if c in returns_df.columns]
completeness = returns_df[all_model_cols].notna().mean().sort_values()

fig = go.Figure(go.Bar(
    x=completeness.values * 100,
    y=completeness.index,
    orientation="h",
    marker_color=[COLORS[0] if v > 0.9 else COLORS[1] if v > 0.5 else COLORS[3] for v in completeness.values],
))
fig.update_layout(
    title="Data Completeness by Model Feature (%)",
    xaxis_title="% Non-Null", yaxis_title="",
    template=PLOTLY_TEMPLATE, height=800, width=900,
)
fig.show()


## 3. Monte Carlo Simulation Analysis


### 3.1 Return Distribution Overview


In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Expected Upside Distribution (MC)",
        "Probability of Positive Return",
        "Value-at-Risk (5%) Distribution",
        "Risk-Reward Ratio Distribution",
    ),
    vertical_spacing=0.12, horizontal_spacing=0.1,
)

upside_clipped = returns_df["expected_upside_mc"].clip(-100, 200)
fig.add_trace(go.Histogram(x=upside_clipped, nbinsx=80, marker_color=COLORS[0], opacity=0.75, name="Upside %"), row=1,
              col=1)
fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=1)
med = returns_df["expected_upside_mc"].median()
fig.add_vline(x=med, line_dash="dot", line_color="green", annotation_text=f"Median: {med:.1f}%", row=1, col=1)

prob_clipped = returns_df["prob_positive_upside"].clip(-100, 100)
fig.add_trace(go.Histogram(x=prob_clipped, nbinsx=60, marker_color=COLORS[2], opacity=0.75, name="P(Positive)"), row=1,
              col=2)
fig.add_vline(x=50, line_dash="dash", line_color="yellow", row=1, col=2)

var5 = returns_df["var_5_pct"].clip(-100, 0)
fig.add_trace(go.Histogram(x=var5, nbinsx=60, marker_color=COLORS[3], opacity=0.75, name="VaR 5%"), row=2, col=1)

rr = returns_df["risk_reward_ratio"].clip(-5, 20)
fig.add_trace(go.Histogram(x=rr, nbinsx=60, marker_color=COLORS[4], opacity=0.75, name="Risk/Reward"), row=2, col=2)
fig.add_vline(x=1, line_dash="dash", line_color="yellow", annotation_text="R/R = 1", row=2, col=2)

fig.update_layout(title="Monte Carlo Simulation: Distribution Overview", template=PLOTLY_TEMPLATE, height=700,
                  width=1100, showlegend=False)
fig.show()


### 3.2 MC Returns by Sector


In [ ]:
mc_sector = (
    returns_df.groupby("sector")
    .agg(
        mean_upside=("expected_upside_mc", "mean"),
        median_upside=("expected_upside_mc", "median"),
        mean_var5=("var_5_pct", "mean"),
        mean_prob_pos=("prob_positive_upside", "mean"),
        mean_rr=("risk_reward_ratio", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_upside", ascending=True)
)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Mean Expected Upside by Sector", "Risk-Reward by Sector"))

fig.add_trace(go.Bar(
    y=mc_sector["sector"], x=mc_sector["mean_upside"], orientation="h",
    marker_color=COLORS[0], name="Mean Upside",
    text=mc_sector["mean_upside"].round(1), textposition="outside",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=mc_sector["mean_var5"], y=mc_sector["mean_upside"],
    mode="markers+text", text=mc_sector["sector"], textposition="top center",
    marker=dict(size=mc_sector["count"] / mc_sector["count"].max() * 40 + 8, color=mc_sector["mean_prob_pos"],
                colorscale="RdYlGn", showscale=True, colorbar=dict(title="P(Pos)")),
    name="Sectors",
), row=1, col=2)

fig.update_layout(template=PLOTLY_TEMPLATE, height=500, width=1200, showlegend=False,
                  title="Monte Carlo: Sector Analysis")
fig.update_xaxes(title_text="Mean Upside (%)", row=1, col=1)
fig.update_xaxes(title_text="Mean VaR 5% (downside)", row=1, col=2)
fig.update_yaxes(title_text="Mean Upside (%)", row=1, col=2)
fig.show()


### 3.3 MC Implied Return vs Price Target Spread


In [ ]:
fig = px.scatter(
    returns_df.dropna(subset=["implied_return_mc", "pt_spread"]).sample(min(2000, len(returns_df)), random_state=42),
    x="pt_spread", y="implied_return_mc",
    color="sector", size="prob_positive_upside",
    hover_data=["ticker", "name"],
    title="MC Implied Return vs Price Target Spread",
    template=PLOTLY_TEMPLATE,
    opacity=0.6,
)
fig.update_layout(height=600, width=1000)
fig.show()


## 4. Kalman Filter Model Diagnostics


### 4.1 Kalman Gain & Variance Diagnostics


In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Kalman Gain Distribution",
        "Kalman Variance Distribution",
        "Signal Strength Distribution",
        "Kalman Gain vs Variance",
    ),
)

kg = returns_df["kalman_gain"].dropna()
fig.add_trace(go.Histogram(x=kg.clip(kg.quantile(0.01), kg.quantile(0.99)), nbinsx=60, marker_color=COLORS[5],
                           name="Kalman Gain"), row=1, col=1)

kv = returns_df["kalman_variance"].dropna()
fig.add_trace(go.Histogram(x=np.log10(kv.clip(lower=1e-6)), nbinsx=60, marker_color=COLORS[6], name="log₁₀(Variance)"),
              row=1, col=2)

ss = returns_df["signal_strength"].dropna()
fig.add_trace(go.Histogram(x=ss.clip(ss.quantile(0.01), ss.quantile(0.99)), nbinsx=60, marker_color=COLORS[7],
                           name="Signal Strength"), row=2, col=1)

sample = returns_df.dropna(subset=["kalman_gain", "kalman_variance"]).sample(min(2000, len(returns_df)),
                                                                             random_state=42)
fig.add_trace(go.Scattergl(
    x=sample["kalman_gain"], y=np.log10(sample["kalman_variance"].clip(lower=1e-6)),
    mode="markers", marker=dict(size=3, color=sample["signal_strength"], colorscale="Viridis", showscale=True,
                                colorbar=dict(title="Signal", x=1.02)),
    name="Stocks",
), row=2, col=2)

fig.update_layout(title="Kalman Filter Diagnostics", template=PLOTLY_TEMPLATE, height=700, width=1100, showlegend=False)
fig.update_xaxes(title_text="Kalman Gain", row=2, col=2)
fig.update_yaxes(title_text="log₁₀(Variance)", row=2, col=2)
fig.show()


### 4.2 Kalman vs MC Implied Returns


In [ ]:
both = returns_df.dropna(subset=["implied_return_mc", "implied_return_kalman"]).copy()
both["ir_mc_clip"] = both["implied_return_mc"].clip(-50, 50)
both["ir_kal_clip"] = both["implied_return_kalman"].clip(both["implied_return_kalman"].quantile(0.02),
                                                         both["implied_return_kalman"].quantile(0.98))

corr = both["implied_return_mc"].corr(both["implied_return_kalman"])
print(f"Correlation (MC vs Kalman implied return): {corr:.4f}")

sample = both.sample(min(2000, len(both)), random_state=42)
fig = px.scatter(
    sample, x="ir_mc_clip", y="ir_kal_clip", color="sector",
    hover_data=["ticker", "name"],
    title=f"Implied Return: MC vs Kalman (ρ = {corr:.3f})",
    labels={"ir_mc_clip": "MC Implied Return", "ir_kal_clip": "Kalman Implied Return"},
    template=PLOTLY_TEMPLATE, opacity=0.5,
)
fig.add_shape(type="line", x0=-50, y0=-50, x1=50, y1=50, line=dict(dash="dash", color="white"))
fig.update_layout(height=600, width=900)
fig.show()


## 5. Price Target Achievement Analysis


### 5.1 Achievement Probability Distribution


In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Achievement Probability Distribution", "Achievement by Confidence Level"),
)

ap = returns_df["achievement_probability"].dropna()
fig.add_trace(go.Histogram(x=ap, nbinsx=50, marker_color=COLORS[2], name="Achievement Prob"), row=1, col=1)
fig.add_vline(x=ap.median(), line_dash="dot", line_color="green", annotation_text=f"Median: {ap.median():.1f}%", row=1,
              col=1)

conf_stats = (
    returns_df.dropna(subset=["confidence_level", "achievement_probability"])
    .groupby("confidence_level")["achievement_probability"]
    .agg(["mean", "median", "count"])
    .reset_index()
    .sort_values("mean")
)
fig.add_trace(go.Bar(
    x=conf_stats["confidence_level"], y=conf_stats["mean"],
    marker_color=COLORS[1], name="Mean Achievement",
    text=conf_stats["mean"].round(1), textposition="outside",
), row=1, col=2)

fig.update_layout(title="Price Target Achievement Analysis", template=PLOTLY_TEMPLATE, height=450, width=1100,
                  showlegend=False)
fig.show()


### 5.2 Analyst Conviction vs Achievement


In [ ]:
sample = returns_df.dropna(subset=["analyst_conviction", "achievement_probability"]).sample(min(2000, len(returns_df)),
                                                                                            random_state=42)
fig = px.scatter(
    sample, x="analyst_conviction", y="achievement_probability",
    color="sector", size="bullish_pct",
    hover_data=["ticker", "name"],
    title="Analyst Conviction vs Price Target Achievement Probability",
    template=PLOTLY_TEMPLATE, opacity=0.5,
)
fig.update_layout(height=600, width=1000)
fig.show()


## 6. Earnings Beat Probability Analysis


### 6.1 Posterior Beat Probability Distribution


In [ ]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Posterior Beat Probability", "By Beat Classification", "Confidence Score vs Beat Prob"),
)

bp = returns_df["posterior_beat_prob"].dropna()
fig.add_trace(go.Histogram(x=bp, nbinsx=50, marker_color=COLORS[8], name="P(Beat)"), row=1, col=1)
fig.add_vline(x=0.5, line_dash="dash", line_color="red", row=1, col=1)

beat_stats = (
    returns_df[returns_df["beat_classification"].isin(["likely_beat", "uncertain", "likely_miss"])]
    .groupby("beat_classification")["posterior_beat_prob"]
    .agg(["mean", "count"])
    .reset_index()
)
fig.add_trace(go.Bar(
    x=beat_stats["beat_classification"], y=beat_stats["mean"],
    marker_color=[COLORS[2], COLORS[1], COLORS[3]], name="Mean P(Beat)",
    text=beat_stats["mean"].round(3), textposition="outside",
), row=1, col=2)

sample = returns_df.dropna(subset=["confidence_score", "posterior_beat_prob"]).sample(min(2000, len(returns_df)),
                                                                                      random_state=42)
fig.add_trace(go.Scattergl(
    x=sample["confidence_score"], y=sample["posterior_beat_prob"],
    mode="markers", marker=dict(size=3, color=sample["momentum_signal"], colorscale="RdYlGn", showscale=True,
                                colorbar=dict(title="Momentum")),
    name="Stocks",
), row=1, col=3)

fig.update_layout(title="Earnings Beat Probability Analysis", template=PLOTLY_TEMPLATE, height=450, width=1300,
                  showlegend=False)
fig.show()


### 6.2 Technical Analysis Signals


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Momentum Signal Distribution", "Volatility Regime Score"))

ms = returns_df["momentum_signal"].dropna()
fig.add_trace(
    go.Histogram(x=ms.clip(ms.quantile(0.02), ms.quantile(0.98)), nbinsx=60, marker_color=COLORS[9], name="Momentum"),
    row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="white", row=1, col=1)

vr = returns_df["volatility_regime_score"].dropna()
fig.add_trace(go.Histogram(x=vr.clip(vr.quantile(0.02), vr.quantile(0.98)), nbinsx=60, marker_color=COLORS[10],
                           name="Vol Regime"), row=1, col=2)

fig.update_layout(title="Earnings: Momentum & Volatility Regime", template=PLOTLY_TEMPLATE, height=400, width=1000,
                  showlegend=False)
fig.show()


## 7. Credit Risk Analysis


### 7.1 Distress Probability & Altman Z-Score


In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Distress Probability Distribution",
        "Altman Z-Score Distribution",
        "Survival Probability Distribution",
        "Distress Probability vs Altman Z-Score",
    ),
)

dp = returns_df["distress_probability"].dropna()
fig.add_trace(go.Histogram(x=dp, nbinsx=50, marker_color=COLORS[3], name="P(Distress)"), row=1, col=1)

az = returns_df["altman_z_score"].dropna()
az_clip = az.clip(az.quantile(0.02), az.quantile(0.98))
fig.add_trace(go.Histogram(x=az_clip, nbinsx=60, marker_color=COLORS[4], name="Altman Z"), row=1, col=2)
fig.add_vline(x=1.81, line_dash="dash", line_color="red", annotation_text="Distress < 1.81", row=1, col=2)
fig.add_vline(x=2.99, line_dash="dash", line_color="green", annotation_text="Safe > 2.99", row=1, col=2)

sp = returns_df["survival_probability"].dropna()
fig.add_trace(go.Histogram(x=sp, nbinsx=50, marker_color=COLORS[2], name="P(Survival)"), row=2, col=1)

sample = returns_df.dropna(subset=["distress_probability", "altman_z_score"]).sample(min(2000, len(returns_df)),
                                                                                     random_state=42)
fig.add_trace(go.Scattergl(
    x=sample["altman_z_score"].clip(-5, 20), y=sample["distress_probability"],
    mode="markers", marker=dict(size=3, color=COLORS[5]),
    name="Stocks",
), row=2, col=2)

fig.update_layout(title="Credit Risk Diagnostics", template=PLOTLY_TEMPLATE, height=700, width=1100, showlegend=False)
fig.update_xaxes(title_text="Altman Z-Score", row=2, col=2)
fig.update_yaxes(title_text="P(Distress)", row=2, col=2)
fig.show()


### 7.2 Credit Risk by Sector


In [ ]:
credit_sector = (
    returns_df.groupby("sector")
    .agg(
        mean_distress=("distress_probability", "mean"),
        mean_altman=("altman_z_score", "mean"),
        mean_survival=("survival_probability", "mean"),
        mean_liquidity=("liquidity_stress_score", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_distress", ascending=True)
)

fig = px.bar(
    credit_sector, x="sector", y="mean_distress",
    color="mean_altman", color_continuous_scale="RdYlGn",
    text=credit_sector["mean_distress"].round(3),
    title="Mean Distress Probability by Sector (color = Altman Z)",
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(height=500, width=1000)
fig.show()


## 8. Dividend Safety Analysis


### 8.1 Dividend Cut Probability & Coverage


In [ ]:
div_df = returns_df.dropna(subset=["dividend_cut_probability"]).copy()
print(f"Stocks with dividend data: {len(div_df):,}")

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Dividend Cut Probability", "FCF Dividend Coverage", "Payout Ratio"),
)

fig.add_trace(go.Histogram(x=div_df["dividend_cut_probability"], nbinsx=50, marker_color=COLORS[3], name="P(Cut)"),
              row=1, col=1)

fcf = div_df["fcf_dividend_coverage"].dropna().clip(-5, 20)
fig.add_trace(go.Histogram(x=fcf, nbinsx=60, marker_color=COLORS[2], name="FCF Coverage"), row=1, col=2)
fig.add_vline(x=1, line_dash="dash", line_color="red", annotation_text="Coverage = 1x", row=1, col=2)

pr = div_df["payout_ratio"].dropna().clip(0, 200)
fig.add_trace(go.Histogram(x=pr, nbinsx=60, marker_color=COLORS[1], name="Payout Ratio"), row=1, col=3)
fig.add_vline(x=100, line_dash="dash", line_color="red", row=1, col=3)

fig.update_layout(title="Dividend Safety Diagnostics", template=PLOTLY_TEMPLATE, height=400, width=1300,
                  showlegend=False)
fig.show()


## 9. Accounting Anomaly Detection


### 9.1 Anomaly Score Distribution & Sector Comparison


In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Accounting Anomaly Score Distribution", "Sector-Relative Anomaly by Sector"),
)

aas = returns_df["accounting_anomaly_score"].dropna()
fig.add_trace(go.Histogram(x=aas, nbinsx=50, marker_color=COLORS[11], name="Anomaly Score"), row=1, col=1)

anomaly_sector = (
    returns_df.groupby("sector")["sector_relative_anomaly"]
    .agg(["mean", "median"])
    .reset_index()
    .sort_values("mean")
)
fig.add_trace(go.Bar(
    x=anomaly_sector["sector"], y=anomaly_sector["mean"],
    marker_color=COLORS[12], name="Mean Sector-Relative",
    text=anomaly_sector["mean"].round(2), textposition="outside",
), row=1, col=2)

fig.update_layout(title="Accounting Anomaly Analysis", template=PLOTLY_TEMPLATE, height=450, width=1100,
                  showlegend=False)
fig.show()


### 9.2 Anomaly Flags Heatmap


In [ ]:
flag_cols = [
    "accumulated_deficit_flag", "negative_wc_flag", "wc_deteriorating_flag",
    "intangibles_growth_flag", "inventory_buildup_flag", "has_goodwill_impairment",
    "has_asset_writedown", "has_restructuring", "has_unusual_items_flag",
    "low_tax_flag", "overinvestment_flag", "recent_acquisition_flag",
    "high_rnd_intensity_flag", "layoff_risk_flag", "repeat_offender_flag",
]
valid_flags = [c for c in flag_cols if c in returns_df.columns]

flag_by_sector = returns_df.groupby("sector")[valid_flags].mean()

fig = go.Figure(go.Heatmap(
    z=flag_by_sector.values,
    x=[c.replace("_flag", "").replace("_", " ").title() for c in flag_by_sector.columns],
    y=flag_by_sector.index,
    colorscale="YlOrRd",
    text=np.round(flag_by_sector.values, 2),
    texttemplate="%{text}",
))
fig.update_layout(
    title="Accounting Anomaly Flag Prevalence by Sector",
    template=PLOTLY_TEMPLATE, height=500, width=1200,
    xaxis=dict(tickangle=45),
)
fig.show()


## 10. Cross-Model Agreement & Composite Analysis


### 10.1 Model Agreement Distribution


In [ ]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Composite Score Distribution", "Weighted Agreement Distribution", "Signal Distribution"),
)

cs = returns_df["composite_score"].dropna()
fig.add_trace(go.Histogram(x=cs, nbinsx=50, marker_color=COLORS[0], name="Composite"), row=1, col=1)
fig.add_vline(x=cs.median(), line_dash="dot", line_color="green", annotation_text=f"Median: {cs.median():.1f}", row=1,
              col=1)

wa = returns_df["weighted_agreement"].dropna()
fig.add_trace(go.Histogram(x=wa, nbinsx=50, marker_color=COLORS[2], name="Weighted Agr."), row=1, col=2)

sig_counts = returns_df["signal"].value_counts()
fig.add_trace(go.Bar(
    x=sig_counts.index, y=sig_counts.values,
    marker_color=[COLORS[2] if "true" in str(s).lower() else COLORS[3] for s in sig_counts.index],
    name="Signal",
), row=1, col=3)

fig.update_layout(title="Cross-Model Agreement Analysis", template=PLOTLY_TEMPLATE, height=400, width=1300,
                  showlegend=False)
fig.show()


### 10.2 Tri-Model Implied Return Comparison


In [ ]:
tri = returns_df.dropna(subset=["implied_return_mc", "implied_return_kalman", "implied_return_pt"]).copy()
print(f"Stocks with all 3 model returns: {len(tri):,}")

# Correlation matrix
corr_cols = ["implied_return_mc", "implied_return_kalman", "implied_return_pt"]
corr_matrix = tri[corr_cols].corr()
print("\nImplied Return Correlation Matrix:")
display(corr_matrix.round(4))

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=["MC", "Kalman", "Price Target"],
    y=["MC", "Kalman", "Price Target"],
    colorscale="RdBu", zmid=0,
    text=np.round(corr_matrix.values, 3),
    texttemplate="%{text}",
))
fig.update_layout(title="Implied Return Correlation: MC vs Kalman vs Price Target", template=PLOTLY_TEMPLATE,
                  height=400, width=500)
fig.show()


### 10.3 Z-Score & Percentile Rank Comparison


In [ ]:
zscore_cols = ["implied_return_mc_zscore", "implied_return_kalman_zscore", "implied_return_pt_zscore"]
valid_zs = [c for c in zscore_cols if c in returns_df.columns]

fig = go.Figure()
for i, col in enumerate(valid_zs):
    vals = returns_df[col].dropna().clip(-4, 4)
    fig.add_trace(go.Histogram(x=vals, nbinsx=60, opacity=0.5,
                               name=col.replace("implied_return_", "").replace("_zscore", "").upper(),
                               marker_color=COLORS[i]))

fig.update_layout(
    title="Implied Return Z-Scores Across Models",
    xaxis_title="Z-Score", yaxis_title="Count",
    barmode="overlay", template=PLOTLY_TEMPLATE, height=450, width=900,
)
fig.show()


In [ ]:
pctile_cols = ["implied_return_mc_pctile", "implied_return_kalman_pctile", "implied_return_pt_pctile"]
valid_pc = [c for c in pctile_cols if c in returns_df.columns]

if valid_pc:
    pctile_corr = returns_df[valid_pc].corr()
    print("Percentile Rank Correlation:")
    display(pctile_corr.round(4))

    sample = returns_df.dropna(subset=valid_pc).sample(min(2000, len(returns_df)), random_state=42)
    fig = px.scatter_matrix(
        sample, dimensions=valid_pc,
        labels={c: c.replace("implied_return_", "").replace("_pctile", " %ile") for c in valid_pc},
        title="Percentile Rank Scatter Matrix",
        template=PLOTLY_TEMPLATE, opacity=0.3,
    )
    fig.update_layout(height=700, width=700)
    fig.show()


## 11. Composite Score Deep Dive


### 11.1 Composite Score by Sector & Size


In [ ]:
comp_sector = (
    returns_df.groupby(["sector", "size_class"])["composite_score"]
    .agg(["mean", "median", "count"])
    .reset_index()
)
comp_sector = comp_sector[comp_sector["size_class"].isin(["Small Cap", "Mid Cap", "Large Cap"])]

fig = px.bar(
    comp_sector, x="sector", y="mean", color="size_class",
    barmode="group",
    title="Mean Composite Score by Sector & Size Class",
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLORS[0], COLORS[2], COLORS[4]],
)
fig.update_layout(height=500, width=1100, xaxis_tickangle=45)
fig.show()


### 11.2 Top & Bottom Stocks by Composite Score


In [ ]:
display_cols = ["ticker", "name", "sector", "size_class", "composite_score", "implied_return_mc",
                "implied_return_kalman", "implied_return_pt", "achievement_probability", "posterior_beat_prob",
                "distress_probability"]
valid_display = [c for c in display_cols if c in returns_df.columns]

top_20 = returns_df.nlargest(20, "composite_score")[valid_display]
print("🏆 Top 20 Stocks by Composite Score:")
display(top_20)

bottom_20 = returns_df.nsmallest(20, "composite_score")[valid_display]
print("\n⚠️ Bottom 20 Stocks by Composite Score:")
display(bottom_20)


## 12. Risk-Return Frontier


In [ ]:
frontier_df = returns_df.dropna(subset=["implied_return_mc", "upside_std", "composite_score"]).copy()
frontier_df["ir_clip"] = frontier_df["implied_return_mc"].clip(-30, 50)
frontier_df["std_clip"] = frontier_df["upside_std"].clip(0, frontier_df["upside_std"].quantile(0.98))

sample = frontier_df.sample(min(3000, len(frontier_df)), random_state=42)
fig = px.scatter(
    sample, x="std_clip", y="ir_clip",
    color="sector", size="composite_score",
    hover_data=["ticker", "name", "composite_score"],
    title="Risk-Return Frontier: MC Implied Return vs Upside Volatility",
    labels={"std_clip": "Upside Std Dev (%)", "ir_clip": "MC Implied Return (%)"},
    template=PLOTLY_TEMPLATE, opacity=0.5,
)
# Efficient frontier approximation
for q in [0.25, 0.5, 0.75]:
    std_bins = pd.qcut(frontier_df["std_clip"], 20, duplicates="drop")
    frontier_line = frontier_df.groupby(std_bins)["ir_clip"].quantile(q).reset_index(level=0, drop=True)

fig.update_layout(height=650, width=1100)
fig.show()


## 13. Model Diagnostics Summary


In [ ]:
# Summary statistics table
diag = pd.DataFrame({
    "Model": ["Monte Carlo", "Kalman Filter", "Price Target", "Earnings Beat", "Credit Risk", "Dividend Safety",
              "Accounting Anomaly"],
    "Coverage (%)": [
        returns_df["implied_return_mc"].notna().mean() * 100,
        returns_df["implied_return_kalman"].notna().mean() * 100,
        returns_df["implied_return_pt"].notna().mean() * 100,
        returns_df["posterior_beat_prob"].notna().mean() * 100,
        returns_df["distress_probability"].notna().mean() * 100,
        returns_df["dividend_cut_probability"].notna().mean() * 100,
        returns_df["accounting_anomaly_score"].notna().mean() * 100,
    ],
    "Mean": [
        returns_df["implied_return_mc"].mean(),
        returns_df["implied_return_kalman"].mean(),
        returns_df["implied_return_pt"].mean(),
        returns_df["posterior_beat_prob"].mean(),
        returns_df["distress_probability"].mean(),
        returns_df["dividend_cut_probability"].mean(),
        returns_df["accounting_anomaly_score"].mean(),
    ],
    "Median": [
        returns_df["implied_return_mc"].median(),
        returns_df["implied_return_kalman"].median(),
        returns_df["implied_return_pt"].median(),
        returns_df["posterior_beat_prob"].median(),
        returns_df["distress_probability"].median(),
        returns_df["dividend_cut_probability"].median(),
        returns_df["accounting_anomaly_score"].median(),
    ],
    "Std": [
        returns_df["implied_return_mc"].std(),
        returns_df["implied_return_kalman"].std(),
        returns_df["implied_return_pt"].std(),
        returns_df["posterior_beat_prob"].std(),
        returns_df["distress_probability"].std(),
        returns_df["dividend_cut_probability"].std(),
        returns_df["accounting_anomaly_score"].std(),
    ],
}).round(4)

print("📊 Model Diagnostics Summary")
display(diag)


In [ ]:
# Cross-model correlation heatmap (all key outputs)
key_outputs = [
    "implied_return_mc", "implied_return_kalman", "implied_return_pt",
    "posterior_beat_prob", "distress_probability", "dividend_cut_probability",
    "accounting_anomaly_score", "composite_score", "achievement_probability",
    "risk_reward_ratio", "prob_positive_upside",
]
valid_outputs = [c for c in key_outputs if c in returns_df.columns]
full_corr = returns_df[valid_outputs].corr()

labels = [c.replace("implied_return_", "IR_").replace("_probability", "_prob").replace("accounting_", "acct_") for c in
          valid_outputs]

fig = go.Figure(go.Heatmap(
    z=full_corr.values,
    x=labels, y=labels,
    colorscale="RdBu", zmid=0,
    text=np.round(full_corr.values, 2),
    texttemplate="%{text}",
))
fig.update_layout(
    title="Cross-Model Output Correlation Matrix",
    template=PLOTLY_TEMPLATE, height=600, width=700,
)
fig.show()


## 14. Sector-Level Model Comparison Dashboard


In [ ]:
sector_summary = (
    returns_df.groupby("sector")
    .agg(
        n=("ticker", "count"),
        mc_return=("implied_return_mc", "median"),
        kalman_return=("implied_return_kalman", "median"),
        pt_return=("implied_return_pt", "median"),
        beat_prob=("posterior_beat_prob", "median"),
        distress=("distress_probability", "median"),
        div_cut=("dividend_cut_probability", "median"),
        anomaly=("accounting_anomaly_score", "median"),
        composite=("composite_score", "median"),
    )
    .reset_index()
    .sort_values("composite", ascending=False)
)

print("📋 Sector-Level Model Summary (Medians)")
display(sector_summary.round(4))

# Radar chart for top 5 sectors
top_sectors = sector_summary.head(5)
radar_cols = ["mc_return", "kalman_return", "pt_return", "beat_prob", "composite"]

fig = go.Figure()
for _, row in top_sectors.iterrows():
    vals = [row[c] for c in radar_cols]
    # Normalize to 0-1 for radar
    mins = sector_summary[radar_cols].min()
    maxs = sector_summary[radar_cols].max()
    norm_vals = [(v - mn) / (mx - mn + 1e-9) for v, mn, mx in zip(vals, mins, maxs)]
    norm_vals.append(norm_vals[0])  # close the polygon
    fig.add_trace(go.Scatterpolar(
        r=norm_vals,
        theta=["MC Return", "Kalman Return", "PT Return", "Beat Prob", "Composite", "MC Return"],
        name=row["sector"],
        fill="toself", opacity=0.3,
    ))

fig.update_layout(
    title="Top 5 Sectors: Normalized Model Output Radar",
    template=PLOTLY_TEMPLATE, height=550, width=700,
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
)
fig.show()


## 15. Key Findings & Observations


In [ ]:
total = len(returns_df)
bullish_pct = (returns_df["signal"] == "true").sum() / total * 100
high_composite = (returns_df["composite_score"] > returns_df["composite_score"].quantile(0.75)).sum()
low_distress = (returns_df["distress_probability"] < 0.3).sum()
high_beat = (returns_df["posterior_beat_prob"] > 0.5).sum()

print("=" * 60)
print("  KEY FINDINGS")
print("=" * 60)
print(f"  Total stocks analyzed:           {total:,}")
print(f"  Bullish signal:                  {bullish_pct:.1f}%")
print(f"  Top quartile composite score:    {high_composite:,} stocks")
print(f"  Low distress (P < 0.3):          {low_distress:,} stocks ({low_distress / total * 100:.1f}%)")
print(f"  Likely earnings beat (P > 0.5):  {high_beat:,} stocks ({high_beat / total * 100:.1f}%)")
print(f"  MC median implied return:        {returns_df['implied_return_mc'].median():.2f}%")
print(f"  Kalman median implied return:    {returns_df['implied_return_kalman'].median():.2f}")
print(f"  PT median achievement prob:      {returns_df['achievement_probability'].median():.1f}%")
print("=" * 60)
